In [47]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

print("Imports completed successfully!")

Imports completed successfully!


In [48]:
train_df = pd.read_parquet("../data/train.parquet")
val_df = pd.read_parquet("../data/validation.parquet")
test_df = pd.read_parquet("../data/test.parquet")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Training: (58516, 45)
Validation: (19506, 45)
Test: (19506, 45)


In [49]:
# Starting features from M1 and M2
features = ["price", "windows", "mac", "linux"]

# Target: positive reviews on a logarithmic scale
target = np.log1p(train_df["positive"])

# Prepare training data
X_train = train_df[features]
y_train = target

# Prepare validation data
X_val = val_df[features]
y_val = np.log1p(val_df["positive"])

print("Features:", features)
print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)

Features: ['price', 'windows', 'mac', 'linux']
Training data: (58516, 4)
Validation data: (19506, 4)


In [50]:
# Baseline model
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)

# Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Make predictions on validation data
dummy_predictions = dummy_model.predict(X_val)
linear_predictions = linear_model.predict(X_val)

# Calculate evaluation metrics
results = pd.DataFrame({
    "Model": ["Dummy Regressor", "Linear Regression"],
    "MAE": [
        mean_absolute_error(y_val, dummy_predictions),
        mean_absolute_error(y_val, linear_predictions)
    ],
    "R²": [
        r2_score(y_val, dummy_predictions),
        r2_score(y_val, linear_predictions)
    ]
})

results.round(4)

,Model,MAE,R²
0,Dummy Regressor,1.9236,-0.0000
1,Linear Regression,1.8862,0.0332


In [51]:
# Potential additional features for M3

candidate_features = [
    "required_age",
    "dlc_count",
    "achievements",
    "supported_languages",
    "genres",
    "categories",
    "tags",
    "release_year"
]

# Inspect data types, missing values and example values

for feature in candidate_features:
    print(f"\n--- {feature} ---")
    print("Data type:", train_df[feature].dtype)
    print("Missing values:", train_df[feature].isna().sum())
    print("Example values:")
    print(train_df[feature].head(3).tolist())


--- required_age ---
Data type: int64
Missing values: 0
Example values:
[0, 0, 0]

--- dlc_count ---
Data type: int64
Missing values: 0
Example values:
[0, 1, 0]

--- achievements ---
Data type: int64
Missing values: 0
Example values:
[0, 24, 18]

--- supported_languages ---
Data type: object
Missing values: 0
Example values:
[array(['English'], dtype=object), array(['English', 'Japanese', 'Simplified Chinese', 'Traditional Chinese',
       'Korean', 'German', 'Spanish - Spain', 'Russian',
       'Portuguese - Brazil'], dtype=object), array(['English', 'German', 'Hungarian', 'Romanian', 'Russian',
       'Spanish - Spain', 'Spanish - Latin America', 'Turkish',
       'Ukrainian'], dtype=object)]

--- genres ---
Data type: object
Missing values: 0
Example values:
[array(['Adventure', 'Indie', 'Free To Play'], dtype=object), array(['Action', 'Indie'], dtype=object), array(['Adventure', 'Indie'], dtype=object)]

--- categories ---
Data type: object
Missing values: 0
Example values:
[arra

In [52]:
# Inspect numerical feature candidates

numeric_candidates = [
    "required_age",
    "dlc_count",
    "achievements",
    "release_year"
]

print(train_df[numeric_candidates].describe().T)

                count         mean         std     min     25%     50%  \
required_age  58516.0     0.196237    1.786549     0.0     0.0     0.0   
dlc_count     58516.0     0.640714   17.495939     0.0     0.0     0.0   
achievements  58516.0    18.668569  142.450690     0.0     0.0     3.0   
release_year  58516.0  2020.651753    3.075488  2006.0  2019.0  2021.0   

                 75%     max  
required_age     0.0    18.0  
dlc_count        0.0  3703.0  
achievements    19.0  5394.0  
release_year  2023.0  2024.0  


In [53]:
# Check correlations using training data only

correlation_df = train_df[
    ["required_age", "release_year", "dlc_count", "achievements"]
].copy()

correlation_df["log_positive"] = np.log1p(train_df["positive"])

correlations = correlation_df.corr()["log_positive"].drop("log_positive")

print("Correlation with log-transformed positive reviews:")
print(correlations.sort_values(ascending=False).round(4))

Correlation with log-transformed positive reviews:
required_age    0.1878
achievements    0.0915
dlc_count       0.0538
release_year   -0.4142
Name: log_positive, dtype: float64


### Feature selection considerations

Some features were excluded from the final model because they may contain information that was only available after a game was released.

- `dlc_count`: The number of DLCs can increase after release.
- `achievements`: Achievements can be added or changed after release.
- `tags`: Steam user tags can be added and modified by users over time.
- `supported_languages_count`: Adding this feature resulted in almost no improvement to the validation metrics.

I kept `required_age` and `release_year` because they are relevant release-time information. The number of categories and genres improved the model, although the current dataset does not guarantee that these values are identical to their original pre-release versions.

In [54]:
# Compare the original model with additional features

base_features = ["price", "windows", "mac", "linux"]
candidate_features = ["required_age", "release_year"]

results = []

# Original model
base_model = LinearRegression()
base_model.fit(train_df[base_features], y_train)

base_predictions = base_model.predict(val_df[base_features])

results.append({
    "Features": "Original M1/M2 features",
    "MAE": mean_absolute_error(y_val, base_predictions),
    "R2": r2_score(y_val, base_predictions)
})

# Add one candidate feature at a time
for feature in candidate_features:

    selected_features = base_features + [feature]

    model = LinearRegression()
    model.fit(train_df[selected_features], y_train)

    predictions = model.predict(val_df[selected_features])

    results.append({
        "Features": "Base + " + feature,
        "MAE": mean_absolute_error(y_val, predictions),
        "R2": r2_score(y_val, predictions)
    })

results_df = pd.DataFrame(results)

print(results_df.round(4))

                  Features     MAE      R2
0  Original M1/M2 features  1.8862  0.0332
1      Base + required_age  1.8662  0.0611
2      Base + release_year  1.7013  0.1886


In [55]:
# Count the number of entries in list-based features

list_features = [
    "genres",
    "categories",
    "supported_languages"
]

for feature in list_features:
    print(f"\n{feature}")
    print("Minimum:", train_df[feature].apply(len).min())
    print("Median:", train_df[feature].apply(len).median())
    print("Maximum:", train_df[feature].apply(len).max())


genres
Minimum: 0
Median: 3.0
Maximum: 16

categories
Minimum: 0
Median: 3.0
Maximum: 29

supported_languages
Minimum: 0
Median: 1.0
Maximum: 103


In [56]:
# Test whether list counts improve the current best model

base_features = [
    "price",
    "windows",
    "mac",
    "linux",
    "required_age",
    "release_year"
]

results = []

# Current best model
model = LinearRegression()
model.fit(train_df[base_features], y_train)

predictions = model.predict(val_df[base_features])

results.append({
    "Features": "Current best model",
    "MAE": mean_absolute_error(y_val, predictions),
    "R2": r2_score(y_val, predictions)
})

# Test each list-based feature separately
for feature in ["genres", "categories", "supported_languages"]:

    train_count = train_df[feature].apply(len)
    val_count = val_df[feature].apply(len)

    X_train_test = train_df[base_features].copy()
    X_val_test = val_df[base_features].copy()

    X_train_test[feature + "_count"] = train_count
    X_val_test[feature + "_count"] = val_count

    model = LinearRegression()
    model.fit(X_train_test, y_train)

    predictions = model.predict(X_val_test)

    results.append({
        "Features": "Current best + " + feature,
        "MAE": mean_absolute_error(y_val, predictions),
        "R2": r2_score(y_val, predictions)
    })

results_df = pd.DataFrame(results)

print(results_df.round(4))

                             Features     MAE      R2
0                  Current best model  1.6913  0.2037
1               Current best + genres  1.6770  0.2113
2           Current best + categories  1.5657  0.3150
3  Current best + supported_languages  1.6893  0.2056


In [57]:
# Test whether genres and supported languages improve the categories model

base_features = [
    "price",
    "windows",
    "mac",
    "linux",
    "required_age",
    "release_year"
]

# Prepare data with category count
X_train_categories = train_df[base_features].copy()
X_val_categories = val_df[base_features].copy()

X_train_categories["categories_count"] = train_df["categories"].apply(len)
X_val_categories["categories_count"] = val_df["categories"].apply(len)

results = []

# Categories model
model = LinearRegression()
model.fit(X_train_categories, y_train)

predictions = model.predict(X_val_categories)

results.append({
    "Features": "Base + categories",
    "MAE": mean_absolute_error(y_val, predictions),
    "R2": r2_score(y_val, predictions)
})

# Add genres
X_train_genres = X_train_categories.copy()
X_val_genres = X_val_categories.copy()

X_train_genres["genres_count"] = train_df["genres"].apply(len)
X_val_genres["genres_count"] = val_df["genres"].apply(len)

model = LinearRegression()
model.fit(X_train_genres, y_train)

predictions = model.predict(X_val_genres)

results.append({
    "Features": "Base + categories + genres",
    "MAE": mean_absolute_error(y_val, predictions),
    "R2": r2_score(y_val, predictions)
})

# Add supported languages instead
X_train_languages = X_train_categories.copy()
X_val_languages = X_val_categories.copy()

X_train_languages["supported_languages_count"] = train_df["supported_languages"].apply(len)
X_val_languages["supported_languages_count"] = val_df["supported_languages"].apply(len)

model = LinearRegression()
model.fit(X_train_languages, y_train)

predictions = model.predict(X_val_languages)

results.append({
    "Features": "Base + categories + languages",
    "MAE": mean_absolute_error(y_val, predictions),
    "R2": r2_score(y_val, predictions)
})

print(pd.DataFrame(results).round(4))

                        Features     MAE      R2
0              Base + categories  1.5657  0.3150
1     Base + categories + genres  1.5621  0.3157
2  Base + categories + languages  1.5651  0.3151


In [58]:
# Final selected features

final_features = [
    "price",
    "windows",
    "mac",
    "linux",
    "required_age",
    "release_year",
    "categories_count",
    "genres_count"
]

# Prepare training data
X_train_final = train_df[
    ["price", "windows", "mac", "linux", "required_age", "release_year"]
].copy()

X_train_final["categories_count"] = train_df["categories"].apply(len)
X_train_final["genres_count"] = train_df["genres"].apply(len)

# Prepare validation data
X_val_final = val_df[
    ["price", "windows", "mac", "linux", "required_age", "release_year"]
].copy()

X_val_final["categories_count"] = val_df["categories"].apply(len)
X_val_final["genres_count"] = val_df["genres"].apply(len)

# Train final model
final_model = LinearRegression()
final_model.fit(X_train_final, y_train)

# Validation predictions
final_predictions = final_model.predict(X_val_final)

# Evaluate
print("Final validation MAE:",
      round(mean_absolute_error(y_val, final_predictions), 4))

print("Final validation R2:",
      round(r2_score(y_val, final_predictions), 4))

Final validation MAE: 1.5621
Final validation R2: 0.3157


In [59]:
# Investigate the largest prediction errors

error_analysis = val_df[["name", "positive", "release_year"]].copy()

error_analysis["actual_log_positive"] = y_val.values
error_analysis["predicted_log_positive"] = final_predictions

error_analysis["absolute_error"] = np.abs(
    error_analysis["actual_log_positive"] -
    error_analysis["predicted_log_positive"]
)

# Show the 10 games with the largest errors
largest_errors = error_analysis.sort_values(
    by="absolute_error",
    ascending=False
).head(10)

print(largest_errors.to_string(index=False))

                                               name  positive  release_year  actual_log_positive  predicted_log_positive  absolute_error
                                             MiSide    108883          2024            11.598038                1.500468       10.097570
                                    Content Warning    137483          2024            11.831263                2.240890        9.590373
                                       Ready or Not    210654          2023            12.257977                2.854819        9.403158
                                          ULTRAKILL    165867          2020            12.018948                2.766158        9.252789
                                  People Playground    277746          2019            12.534466                3.547530        8.986936
                                           Soundpad     84918          2017            11.349453                2.627846        8.721607
                                Oxygen No

### Analysis of the largest prediction errors

The largest prediction errors show that the model significantly underestimated several popular games, including MiSide, Content Warning, Ready or Not and ULTRAKILL.

These games received many more positive reviews than the model expected based on their available features. This suggests that the model has difficulty predicting unexpected popular games and breakout successes.

Factors such as viral attention, streaming popularity and community interest may influence a game's success, but these factors are not fully represented by the selected features.

In [60]:
# Prepare test data using the same selected features

X_test_final = test_df[
    ["price", "windows", "mac", "linux", "required_age", "release_year"]
].copy()

X_test_final["categories_count"] = test_df["categories"].apply(len)
X_test_final["genres_count"] = test_df["genres"].apply(len)

# Prepare test target
y_test = np.log1p(test_df["positive"])

# Make predictions
test_predictions = final_model.predict(X_test_final)

# Evaluate final model on the test set
test_mae = mean_absolute_error(y_test, test_predictions)
test_r2 = r2_score(y_test, test_predictions)

print("Final test MAE:", round(test_mae, 4))
print("Final test R2:", round(test_r2, 4))

Final test MAE: 1.5609
Final test R2: 0.3229


In [61]:
# Summary of feature experiments

feature_summary = pd.DataFrame({
    "Model": [
        "Original M1/M2 features",
        "Base + required_age",
        "Base + release_year",
        "Base + required_age + release_year",
        "Base + required_age + release_year + genres",
        "Base + required_age + release_year + categories",
        "Final model: categories + genres"
    ],
    "MAE": [
        1.8862,
        1.8662,
        1.7013,
        1.6913,
        1.6770,
        1.5657,
        1.5621
    ],
    "R2": [
        0.0332,
        0.0611,
        0.1886,
        0.2037,
        0.2113,
        0.3150,
        0.3157
    ]
})

print(feature_summary.round(4).to_string(index=False))

                                          Model    MAE     R2
                        Original M1/M2 features 1.8862 0.0332
                            Base + required_age 1.8662 0.0611
                            Base + release_year 1.7013 0.1886
             Base + required_age + release_year 1.6913 0.2037
    Base + required_age + release_year + genres 1.6770 0.2113
Base + required_age + release_year + categories 1.5657 0.3150
               Final model: categories + genres 1.5621 0.3157


### Conclusion

The goal of M3 was to predict the number of positive Steam reviews using regression. I started with the features from M1 and M2 and compared a Dummy Regressor with Linear Regression. I then tested additional features individually to see whether they improved the validation results.

The final model uses price, platform support, required age, release year, and the number of categories and genres. The final validation MAE was 1.5621 and the R² was 0.3157. When evaluated on the test set, the model achieved an MAE of 1.5609 and an R² of 0.3229.

The results show that the model explains approximately 32% of the variation in log-transformed positive review counts. The error analysis also showed that the model struggled to predict very popular games, often underestimating their review counts.

Overall, the model provides a useful starting point for predicting Steam game popularity, but its accuracy is limited. One limitation is that some metadata, such as categories and genres, may have changed after a game's release. Another limitation is that the model cannot easily predict unexpected breakout successes.